In [1]:
import warnings
import pandas as pd
import os


# Suppress all FutureWarnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Backtest trades function
def backtest_trades(price_data, signal_data, tp=None, sl=None, entry_time_offset=None, percentage_change=None, time_limit_minutes=None):
    output_data = pd.DataFrame(columns=[
        'Datetime', 'Side', 'Signal Open Price', 'Entry Price', 'TP Price', 'SL Price', 'Result', 'Duration', 'Execution Latency', 'ROI', 'NAV', 'Ignore Reason'
    ])
    
    initial_margin = 100000
    current_margin = initial_margin
    exit_datetimes = []

    for i, row in signal_data.iterrows():
        signal_datetime = row['Datetime']
        signal_value = row['Signal']
        
        if signal_value == 0:
            continue
        elif signal_value > 0:
            side = 'Buy'
        else:
            side = 'Sell'
        
        adjusted_signal_datetime = signal_datetime + pd.Timedelta(minutes=entry_time_offset)
        if adjusted_signal_datetime not in price_data.index:
            continue

        signal_open_price = price_data.at[adjusted_signal_datetime, 'Open']
        
        exit_datetimes.sort(key=lambda x: x[0])
        ignore_signal = False
        reason = ''
        if exit_datetimes:
            later_exits = [ed for ed in exit_datetimes if ed[0] > signal_datetime]

            if len(later_exits) >= 3:
                result = 'Ignored'
                reason = 'More than 2 open trades'
                ignore_signal = True
            elif len(later_exits) == 2:
                if later_exits[-1][1] == side:
                    ignore_signal = False
                else:
                    result = 'Ignored'
                    reason = 'Two open trades, last one with different side'
                    ignore_signal = True
            elif len(later_exits) == 1:
                if later_exits[-1][1] != side:
                    ignore_signal = False
                else:
                    result = 'Ignored'
                    reason = 'One open trade with the same side'
                    ignore_signal = True

        if ignore_signal:
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'Signal Open Price': signal_open_price,
                'Entry Price': None,
                'TP Price': None,
                'SL Price': None,
                'Result': result,
                'Duration': '00:00:00',
                'Execution Latency': '00:00:00',
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': reason
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue
        
        entry_datetime, entry_price, entry_duration = determine_entry(price_data, signal_datetime, percentage_change, side, time_limit_minutes, entry_time_offset)
        if entry_datetime is None:
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'Signal Open Price': signal_open_price,
                'Entry Price': None,
                'TP Price': None,
                'SL Price': None,
                'Result': 'Not Filled',
                'Duration': '00:00:00',
                'Execution Latency': '00:00:00',
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': ''
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue
        
        if side == 'Buy':
            tp_price = entry_price * (1 + tp)
            sl_price = entry_price * (1 - sl)
        else:
            tp_price = entry_price * (1 - tp)
            sl_price = entry_price * (1 + sl)
        
        result, duration_str = check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side)
        exit_datetime = entry_datetime + pd.Timedelta(duration_str)

        if result in [1, -1]:
            exit_datetimes.append((exit_datetime, side))

        if result == 1:
            current_margin = current_margin * (1 + tp)
        elif result == -1:
            current_margin = current_margin * (1 - sl)
        
        roi = ((current_margin - initial_margin) / initial_margin) * 100
        nav = current_margin
        initial_margin = current_margin
        
        new_row = pd.DataFrame([{
            'Datetime': signal_datetime,
            'Side': side,
            'Signal Open Price': signal_open_price,
            'Entry Price': entry_price,
            'TP Price': tp_price,
            'SL Price': sl_price,
            'Result': result,
            'Duration': duration_str,
            'Execution Latency': format_duration(entry_duration),
            'ROI': roi,
            'NAV': nav,
            'Ignore Reason': ''
        }])
        
        output_data = pd.concat([output_data, new_row], ignore_index=True)
    
    return output_data

# Helper functions
def format_duration(duration):
    seconds = duration.total_seconds()
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    seconds = int(seconds % 60)
    return f"{hours:02}:{minutes:02}:{seconds:02}"

def check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side):
    result = 0
    exit_datetime = None
    subsequent_prices = price_data.loc[entry_datetime:]

    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy':
            if price_row['High'] >= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['Low'] <= sl_price:
                result = -1
                exit_datetime = current_datetime
                break
        else:
            if price_row['Low'] <= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['High'] >= sl_price:
                result = -1
                exit_datetime = current_datetime
                break
    
    if exit_datetime:
        duration = exit_datetime - entry_datetime
        duration_str = format_duration(duration)
    else:
        duration_str = '00:00:00'
        
    return result, duration_str

def determine_entry(price_data, signal_datetime, percentage_change, side, time_limit_minutes, entry_time_offset):
    adjusted_signal_datetime = signal_datetime + pd.Timedelta(minutes=entry_time_offset)
    if adjusted_signal_datetime not in price_data.index:
        return None, None, None
    
    adjusted_open_price = price_data.at[adjusted_signal_datetime, 'Open']
    percentage_change_price = adjusted_open_price * (1 - percentage_change) if side == 'Buy' else adjusted_open_price * (1 + percentage_change)
    
    time_limit = adjusted_signal_datetime + pd.Timedelta(minutes=time_limit_minutes)
    subsequent_prices = price_data.loc[adjusted_signal_datetime:time_limit]
    
    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy' and price_row['Low'] <= percentage_change_price:
            entry_price = percentage_change_price
            duration = current_datetime - adjusted_signal_datetime
            return current_datetime, entry_price, duration
        elif side == 'Sell' and price_row['High'] >= percentage_change_price:
            entry_price = percentage_change_price
            duration = current_datetime - adjusted_signal_datetime
            return current_datetime, entry_price, duration

    return None, None, None

def calculate_metrics(group, initial_nav):
    total_trades = len(group[(group['Result'] == 1) | (group['Result'] == -1)])
    total_wins = len(group[group['Result'] == 1])
    total_losses = len(group[group['Result'] == -1])
    win_rate = total_wins / total_trades if total_trades > 0 else 0
    
    final_nav = group['NAV'].iloc[-1] if total_trades > 0 else initial_nav
    roi = ((final_nav - initial_nav) / initial_nav) * 100
    
    drawdown = 0
    cumulative_returns = (group['NAV'] - initial_nav).cumsum()
    peak = cumulative_returns.cummax()
    drawdown = (peak - cumulative_returns).max()
    
    return {
        'Total Trades': total_trades,
        'Total Wins': total_wins,
        'Total Losses': total_losses,
        'Win Rate': win_rate,
        'ROI': roi,
        'NAV': final_nav,
        'Max Drawdown': drawdown
    }

def generate_report(price_data, signal_data, scenarios, output_directory, output_file_name):
    report_columns = ['Scenario', 'Period', 'Total Trades', 'Total Wins', 'Total Losses', 'Win Rate', 'ROI', 'NAV', 'Max Drawdown']
    report_data = pd.DataFrame(columns=report_columns)

    for scenario in scenarios:
        tp = scenario['tp']
        sl = scenario['sl']
        percentage_change = scenario.get('percentage_change', None)
        entry_time_offset = scenario.get('entry_time_offset', None)
        time_limit_minutes = scenario.get('time_limit_minutes', None)
        
        # Backtest the trades for the current scenario
        trade_data = backtest_trades(price_data, signal_data, tp, sl, entry_time_offset, percentage_change, time_limit_minutes)
        monthly_groups = trade_data.groupby(trade_data['Datetime'].dt.to_period('M'))
        monthly_reports = []
        initial_nav = 100000
        for month, group in monthly_groups:
            metrics = calculate_metrics(group, initial_nav)
            metrics['Period'] = month.strftime('%Y-%m')
            metrics['Scenario'] = f"TP={tp}, SL={sl}, Offset={entry_time_offset}, pctChange={percentage_change}, TimeLimit={time_limit_minutes}"
            monthly_reports.append(pd.DataFrame([metrics]))
            initial_nav = metrics['NAV']

        if monthly_reports:
            monthly_report = pd.concat(monthly_reports, ignore_index=True)
            report_data = pd.concat([report_data, monthly_report], ignore_index=True)

        overall_metrics = calculate_metrics(trade_data, 100000)
        overall_metrics['Period'] = 'Overall'
        overall_metrics['Scenario'] = f"TP={tp}, SL={sl}, Offset={entry_time_offset}, pctChange={percentage_change}, TimeLimit={time_limit_minutes}"
        overall_report = pd.DataFrame([overall_metrics])
        report_data = pd.concat([report_data, overall_report], ignore_index=True)

    os.makedirs(output_directory, exist_ok=True)
    report_data.to_csv(os.path.join(output_directory, output_file_name), index=False)

    return report_data

In [13]:
price_data = pd.read_csv('E:\Signal Backtesting\Input\Price_2024.csv', parse_dates=['Datetime'],
                         index_col='Datetime')
# Re-load the signal data without setting the index
signal_data = pd.read_csv('E:\Signal Backtesting\Output\Economic_data_analysis\\befor and after news\Filtered_Signal_Before_After_1_Hours.csv', parse_dates=['Datetime'])

In [14]:
# Define the parameters
tp = 0.0096
sl = 0.014
entry_time_offset = 60  # Time offset in minutes
percentage_change = 0.00008
time_limit_minutes = 120

# Call the backtest_trades function
backtest_output = backtest_trades(
    price_data=price_data,
    signal_data=signal_data,
    tp=tp,
    sl=sl,
    entry_time_offset=entry_time_offset,
    percentage_change=percentage_change,
    time_limit_minutes=time_limit_minutes
)

backtest_output

,Datetime,Side,Signal Open Price,Entry Price,TP Price,SL Price,Result,Duration,Execution Latency,ROI,NAV,Ignore Reason
0,2024-01-01 00:00:00,Sell,42503.5,42506.900280,42098.834037,43101.996884,-1,16:59:00,00:13:00,-1.40,98600.000000,
1,2024-01-01 16:00:00,Buy,42759.7,42756.279224,43166.739505,42157.691315,1,01:11:00,00:02:00,0.96,99546.560000,
2,2024-01-02 00:00:00,Sell,45179.7,45183.314376,44749.554558,45815.880777,-1,06:25:00,00:00:00,-1.40,98152.908160,
3,2024-01-02 22:00:00,Buy,45006.7,45003.099464,45435.129219,44373.056072,1,03:08:00,00:00:00,0.96,99095.176078,
4,2024-01-05 00:00:00,Buy,44207.4,44203.863408,44628.220497,43585.009320,-1,00:48:00,00:00:00,-1.40,97707.843613,
...,...,...,...,...,...,...,...,...,...,...,...,...
206,2024-06-27 08:00:00,Sell,60861.3,60866.168904,60281.853683,61718.295269,-1,04:09:00,00:00:00,-1.40,127775.413600,
207,2024-06-28 08:00:00,Buy,61456.4,61451.483488,62041.417729,60591.162719,-1,10:17:00,00:00:00,-1.40,125986.557810,
208,2024-06-28 16:00:00,Sell,60926.9,60931.774152,60346.829120,61784.818990,1,02:04:00,00:13:00,0.96,127196.028764,
209,2024-06-29 03:00:00,Buy,60760.0,60755.139200,61338.388536,59904.567251,1,27:04:00,00:01:00,0.96,128417.110641,


In [11]:
scenario=[
{ 'tp': 0.0096,'sl' : 0.014,'entry_time_offset':60,
'percentage_change' : 0.00008,
'time_limit_minutes' : 120}
]
output_directory ='E:\Signal Backtesting\Output\Economic_data_analysis'
report=generate_report(price_data, signal_data, scenarios=scenario, output_directory=output_directory, output_file_name='Signals_5_hours_before_after_events_analysis.csv')
report

,Scenario,Period,Total Trades,Total Wins,Total Losses,Win Rate,ROI,NAV,Max Drawdown
0,"TP=0.0096, SL=0.014, Offset=60, pctChange=8e-0...",2024-01,26,13,13,0.500000,-5.736982,94263.017690,107396.800937
1,"TP=0.0096, SL=0.014, Offset=60, pctChange=8e-0...",2024-02,25,18,7,0.720000,7.603540,101430.343844,0.000000
2,"TP=0.0096, SL=0.014, Offset=60, pctChange=8e-0...",2024-03,33,23,10,0.696970,8.194208,109741.757438,459.925751
3,"TP=0.0096, SL=0.014, Offset=60, pctChange=8e-0...",2024-04,33,20,13,0.606061,0.782872,110600.895436,51158.616235
4,"TP=0.0096, SL=0.014, Offset=60, pctChange=8e-0...",2024-05,29,20,9,0.689655,6.629917,117933.643363,5585.675321
5,"TP=0.0096, SL=0.014, Offset=60, pctChange=8e-0...",2024-06,27,15,12,0.555556,-2.554204,114921.377590,72621.943817
6,"TP=0.0096, SL=0.014, Offset=60, pctChange=8e-0...",Overall,173,109,64,0.630058,14.921378,114921.377590,156256.313684


In [ ]:
# Define your base scenario
base_scenario = {
    'tp': 0.015,
    'sl': 0.015,
    'percentage_change': 0,
    'time_limit_minutes': 120
}

# Define the list of entry time offsets
entry_time_offsets = [0, 30, 60, 60*2,60*3, 60*4, 60*5, 60*6, 60*7, 60*8, 60*9, 60*10, 60*11, 60*12, 60*13, 60*14, 60*15, 60*16, 60*17, 60*18, 60*19, 60*20, 60*21, 60*22, 60*23, 60*24]

# Define the output directory
output_directory = 'E:\\Signal Backtesting\\Output'

# Loop over the entry time offsets
for offset in entry_time_offsets:
    # Update the entry_time_offset in the scenario
    scenario = [dict(base_scenario, entry_time_offset=offset)]
    
    # Define the output file name
    output_file_name = f'tp=sl=1.5%_backtesting_offset_{offset}.csv'
    
    # Generate the report
    generate_report(price_data, signal_data, scenarios=scenario, output_directory=output_directory, output_file_name=output_file_name)